# Chicago Taxi Trips — week of 11–17 September 2023

A descriptive summary of one week of reported taxi trips: when people ride, where trips start, how they pay and tip, and which companies carry the trips.

**Summary placeholder — replaced at the end of the notebook build.**

---
## 1. Setup and data loading
**Conventions used throughout**
- Every figure is *reported trips*, not total market volume (the City notes not all trips are reported).
- "Revenue" means `trip_total` as reported. It includes tips, tolls and extras, so it is what the rider was charged, not what the company or driver kept.
- Timestamps are local time, rounded to 15 minutes. The brief and `dataset.md` don't state a time zone, so I assume Chicago local time. Durations use `trip_seconds`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

FIG_DIR = Path("figures/analysis")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Chart style: white surface, thin marks, recessive grid, no dual axes.
BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#9a9a96", "#0b0b0b"
plt.rcParams.update({
    "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#52514e", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "font.size": 10,
})

In [2]:
raw = pd.read_csv("trips.csv", dtype={"trip_id": "string", "taxi_id": "string"})
raw["start"] = pd.to_datetime(raw["trip_start_timestamp"])
raw["end"] = pd.to_datetime(raw["trip_end_timestamp"])

print("Rows, columns:", raw.shape, "(brief says 135,768 rows)")
print("Start range  :", raw["start"].min(), "->", raw["start"].max())
print("Duplicate trip_id:", raw["trip_id"].duplicated().sum(), "| fully duplicated rows:", raw.drop(columns=["start", "end"]).duplicated().sum())
print("Trips per calendar date:")
print(raw["start"].dt.date.value_counts().sort_index().to_string())

Rows, columns: (135768, 25) (brief says 135,768 rows)
Start range  : 2023-09-11 00:00:00 -> 2023-09-17 23:45:00
Duplicate trip_id: 0 | fully duplicated rows: 0
Trips per calendar date:
start
2023-09-11    22190
2023-09-12    22040
2023-09-13    22694
2023-09-14    22531
2023-09-15    20226
2023-09-16    12719
2023-09-17    13368


In [3]:
miss = raw.drop(columns=["start", "end"]).isna().sum()
miss = pd.DataFrame({"missing": miss, "pct": miss / len(raw) * 100})
print(miss[miss.missing > 0].to_string())

                            missing   pct
trip_end_timestamp                1  0.00
trip_seconds                     33  0.02
pickup_census_tract           71404 52.59
dropoff_census_tract          73842 54.39
pickup_community_area          3815  2.81
dropoff_community_area        13794 10.16
fare                            324  0.24
tips                            324  0.24
tolls                           324  0.24
extras                          324  0.24
trip_total                      324  0.24
pickup_centroid_latitude       3775  2.78
pickup_centroid_longitude      3775  2.78
pickup_centroid_location       3775  2.78
dropoff_centroid_latitude     12872  9.48
dropoff_centroid_longitude    12872  9.48
dropoff_centroid_location     12872  9.48


---
## 2. Data quality checks and cleaning
First I measure every issue on the raw data; the treatment decisions come after, in a numbered list with row counts.

In [4]:
d = raw.copy()
parts = d["fare"] + d["tips"] + d["tolls"] + d["extras"]
d["gap"] = (d["trip_total"] - parts).round(2)           # trip_total minus the sum of its parts
d["mph"] = d["trip_miles"] / (d["trip_seconds"] / 3600)  # implied speed

checks = {
    "trip_miles == 0": d["trip_miles"].eq(0),
    "trip_miles > 100": d["trip_miles"].gt(100),
    "trip_seconds == 0": d["trip_seconds"].eq(0),
    "trip_seconds > 4 hours": d["trip_seconds"].gt(4 * 3600),
    "trip_seconds missing": d["trip_seconds"].isna(),
    "miles == 0 and seconds == 0": d["trip_miles"].eq(0) & d["trip_seconds"].eq(0),
    "implied speed > 80 mph": d["mph"].gt(80),
    "money fields missing (fare/tips/tolls/extras/total)": d["trip_total"].isna(),
    "fare == 0": d["fare"].eq(0),
    "trip_total == 0": d["trip_total"].eq(0),
    "trip_total > $200": d["trip_total"].gt(200),
    "trip_total > $500": d["trip_total"].gt(500),
    "tolls > $100": d["tolls"].gt(100),
    "extras > $100": d["extras"].gt(100),
    "tips > fare": d["tips"].gt(d["fare"]),
    "trip_total != sum of parts (any gap)": d["gap"].ne(0) & d["gap"].notna(),
}
chk = pd.DataFrame({"rows": {k: int(v.sum()) for k, v in checks.items()}})
chk["pct_of_rows"] = chk["rows"] / len(d) * 100
print(chk.to_string())

                                                      rows  pct_of_rows
trip_miles == 0                                      13993        10.31
trip_miles > 100                                        11         0.01
trip_seconds == 0                                     2731         2.01
trip_seconds > 4 hours                                 121         0.09
trip_seconds missing                                    33         0.02
miles == 0 and seconds == 0                           2637         1.94
implied speed > 80 mph                                 228         0.17
money fields missing (fare/tips/tolls/extras/total)    324         0.24
fare == 0                                              135         0.10
trip_total == 0                                        134         0.10
trip_total > $200                                      109         0.08
trip_total > $500                                        8         0.01
tolls > $100                                             3      

In [5]:
print("Gap = trip_total minus (fare + tips + tolls + extras), rows where gap != 0:")
g = d.loc[d["gap"].ne(0) & d["gap"].notna(), "gap"].value_counts().sort_index()
print(g.to_string())
print("\nNegative gaps (total smaller than parts):", int((d["gap"] < 0).sum()))
print("\nShare of trips with a +$0.50 gap, by payment_type:")
print(d.assign(g50=d["gap"].eq(0.5)).groupby("payment_type")["g50"].agg(["sum", "mean"]).rename(columns={"sum": "trips", "mean": "share"}).to_string())

Gap = trip_total minus (fare + tips + tolls + extras), rows where gap != 0:
gap
0.50    46522
1.00      109
1.50        4
1.95        5
2.00        2
2.50      751
3.00      804

Negative gaps (total smaller than parts): 0

Share of trips with a +$0.50 gap, by payment_type:
              trips  share
payment_type              
Cash              3   0.00
Credit Card   42409   0.70
Dispute           0   0.00
Mobile         4110   0.22
No Charge         0   0.00
Prcard            0   0.00
Unknown           0   0.00


In [6]:
print("Trips with only the $3.25 base fare, zero miles, but a large extra or tip (meter not run?):")
sus = d[(d["trip_miles"] == 0) & (d["fare"] <= 3.5) & ((d["extras"] >= 20) | (d["tips"] >= 5))]
print(len(sus), "rows; median extras", sus["extras"].median(), "| median tips", sus["tips"].median(), "| median total", sus["trip_total"].median())
print("\nLargest 8 trip_total values:")
print(d.nlargest(8, "trip_total")[["trip_miles", "trip_seconds", "fare", "tips", "tolls", "extras", "trip_total", "payment_type"]].to_string())

Trips with only the $3.25 base fare, zero miles, but a large extra or tip (meter not run?):
597 rows; median extras 44.0 | median tips 6.22 | median total 53.25

Largest 8 trip_total values:
        trip_miles  trip_seconds   fare  tips    tolls   extras  trip_total payment_type
124235        0.10      1,080.00  11.25  0.00 4,444.44 4,445.44    8,901.13         Cash
25636         0.00          0.00   3.25  0.00 4,444.44 4,444.44    8,892.13         Cash
35524         1.30        420.00   6.75  0.00 4,111.10     0.44    4,118.29         Cash
127990       37.45           NaN 911.00  0.00     0.00     0.00      911.00         Cash
31648         0.00          3.00 900.00  0.00     0.00     0.00      900.00         Cash
99632         0.00         60.00 758.00  0.00     0.00     0.00      758.00    No Charge
30785         0.00          0.00 550.00  0.00     0.00     0.00      550.00  Credit Card
51799         0.00          0.00 550.00  0.00     0.00     0.00      550.00         Cash
